# Phase 4 — Demand Forecasting
Holt-Winters Exponential Smoothing model for 5 product lines with MAPE evaluation.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../01_cleaned_data/transactions_clean.csv', parse_dates=['date'])

# ── Aggregate to weekly units per product line ───
weekly = (df.groupby([pd.Grouper(key='date', freq='W'), 'product_line'])
            .agg(units=('units_sold', 'sum'),
                 revenue=('revenue_gbp', 'sum'))
            .reset_index())

print(f"Weekly aggregation: {weekly.shape}")
print(f"Product lines: {weekly['product_line'].unique()}")
print(f"Date range: {weekly['date'].min()} to {weekly['date'].max()}")
print(f"Weeks per product line:")
print(weekly.groupby('product_line')['date'].count())

In [ ]:
# ── Holt-Winters Forecasting for all product lines ──
results = []
product_lines = weekly['product_line'].unique()

fig, axes = plt.subplots(len(product_lines), 1, figsize=(14, 4 * len(product_lines)))

for i, pl in enumerate(product_lines):
    # Extract series for this product line
    series = weekly[weekly['product_line'] == pl].set_index('date')['units']
    series = series.asfreq('W', method='ffill')  # Ensure regular frequency
    
    # Train / Test Split — hold last 26 weeks as test
    train = series.iloc[:-26]
    test = series.iloc[-26:]
    
    # Fit Holt-Winters model
    try:
        model = ExponentialSmoothing(
            train,
            trend='add',
            seasonal='add',
            seasonal_periods=52
        ).fit(optimized=True)
        
        # Forecast 13 weeks forward
        test_pred = model.forecast(steps=len(test))
        forecast = model.forecast(steps=13 + len(test))
        
        # Calculate MAPE
        mape = (abs(test - test_pred) / test).mean() * 100
        
        # Store results
        results.append({
            'product_line': pl,
            'mape': round(mape, 1),
            'forecast_13wk': round(forecast.iloc[-13:].mean(), 0)
        })
        
        # Plot
        ax = axes[i]
        train.plot(ax=ax, label='Historical', color='#1a3a5c')
        test.plot(ax=ax, label='Actual (holdout)', color='#2a7a4a')
        test_pred.plot(ax=ax, label='Predicted', color='#c8922a', linestyle='--')
        ax.set_title(f'Demand Forecast — {pl} (MAPE: {mape:.1f}%)')
        ax.legend(loc='upper left')
        
    except Exception as e:
        print(f"Error for {pl}: {e}")
        # Fallback: simpler model without seasonality
        model = ExponentialSmoothing(
            train,
            trend='add',
            seasonal=None
        ).fit(optimized=True)
        
        test_pred = model.forecast(steps=len(test))
        forecast = model.forecast(steps=13 + len(test))
        mape = (abs(test - test_pred) / test).mean() * 100
        
        results.append({
            'product_line': pl,
            'mape': round(mape, 1),
            'forecast_13wk': round(forecast.iloc[-13:].mean(), 0)
        })
        
        ax = axes[i]
        train.plot(ax=ax, label='Historical', color='#1a3a5c')
        test.plot(ax=ax, label='Actual (holdout)', color='#2a7a4a')
        test_pred.plot(ax=ax, label='Predicted (no seasonal)', color='#c8922a', linestyle='--')
        ax.set_title(f'Demand Forecast — {pl} (MAPE: {mape:.1f}%) [fallback]')
        ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('../04_outputs/forecast_all_product_lines.png', dpi=150)
plt.show()

# Print MAPE summary
print("\n" + "=" * 50)
print("FORECAST ACCURACY SUMMARY")
print("=" * 50)
results_df = pd.DataFrame(results)
for _, row in results_df.iterrows():
    status = "🟢" if row['mape'] < 10 else ("🟡" if row['mape'] < 20 else "🔴")
    print(f"  {status} {row['product_line']:25s} MAPE: {row['mape']:6.1f}%   Avg 13wk forecast: {row['forecast_13wk']:,.0f} units")

In [ ]:
# ── Save forecast output for dashboard ───────────
forecast_rows = []

for pl in product_lines:
    series = weekly[weekly['product_line'] == pl].set_index('date')['units']
    series = series.asfreq('W', method='ffill')
    train = series.iloc[:-26]
    
    try:
        model = ExponentialSmoothing(
            train, trend='add', seasonal='add', seasonal_periods=52
        ).fit(optimized=True)
    except:
        model = ExponentialSmoothing(
            train, trend='add', seasonal=None
        ).fit(optimized=True)
    
    forecast = model.forecast(steps=13)
    
    for date, value in forecast.items():
        forecast_rows.append({
            'date': date,
            'product_line': pl,
            'forecast_units': round(max(value, 0), 0)  # No negative forecasts
        })

forecast_df = pd.DataFrame(forecast_rows)
forecast_df.to_csv('../01_cleaned_data/forecasts_output.csv', index=False)

# Also save the MAPE results
results_df.to_csv('../01_cleaned_data/forecast_mape.csv', index=False)

print(f"Forecast saved: {len(forecast_df)} rows")
print(f"\nSample:")
print(forecast_df.head(10))
print(f"\nMAPE saved:")
print(results_df)